In [1]:
%matplotlib qt

Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python\src
Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python


In [53]:
import sys
from pathlib import Path

# ============================================
# PROJECT PATH SETUP
# ============================================

PROJECT_ROOT = Path.cwd().parents[1]   # go from notebooks → Python
SRC_PATH = PROJECT_ROOT / "Python" / "src"

sys.path.append(str(SRC_PATH))

print("Added to path:", SRC_PATH)

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

print("Added to path:", repo_root)


# ============================================
# AIR WHEEL — MP4 DIRECTORY SCANNER
# NeuroMomentum Lab
# ============================================

from pathlib import Path
import json

# --- Windows local data path ---
ROOT = Path(r"E:\Data\UNLV\AIR_Wheel_Methods")

# --- animals and dates of interest ---
# TARGETS = {
#     "NML_M_08": "2026_03_04",
# }

TARGETS = {
    "NML_04_R": "2026_01_24",
}

Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python\src
Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python


In [69]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"E:\Data\UNLV\AIR_Wheel_Methods\PData")

animal = "NML_06_R"
date = "2026_01_16"

session_dir = ROOT / animal / date

videos = list(session_dir.glob("*.mp4"))

print("Videos found:")
for v in videos:
    print(v.name)

Videos found:
face_1440x1080_60_20260116_154145.mp4
pupi_320x240_60_20260116_154145.mp4
video_20260116_154140.mp4


In [70]:
import cv2
import json
from pathlib import Path

def select_and_save_roi(video_path):

    cap = cv2.VideoCapture(str(video_path))
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError("Could not read video frame")

    roi = cv2.selectROI("Select LED ROI", frame, fromCenter=False)
    cv2.destroyAllWindows()

    x, y, w, h = roi

    roi_dict = {
        "x1": int(x),
        "y1": int(y),
        "x2": int(x + w),
        "y2": int(y + h)
    }

    json_path = video_path.with_name(video_path.stem + "_LED_ROI.json")

    with open(json_path, "w") as f:
        json.dump(roi_dict, f, indent=4)

    print("ROI saved:", json_path)

    return roi_dict


def load_roi(video_path):

    json_path = video_path.with_name(video_path.stem + "_LED_ROI.json")

    with open(json_path, "r") as f:
        roi = json.load(f)

    return roi

In [77]:
roi = select_and_save_roi(videos[1])
print("ROI:", roi)

ROI saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\pupi_320x240_60_20260116_154145_LED_ROI.json
ROI: {'x1': 120, 'y1': 20, 'x2': 131, 'y2': 32}


In [74]:
import numpy as np
import pandas as pd

def extract_led_and_save(video_path, roi):

    x1 = roi["x1"]
    y1 = roi["y1"]
    x2 = roi["x2"]
    y2 = roi["y2"]

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        raise RuntimeError("Could not read FPS from video")

    print("Video:", video_path.name, "FPS:", fps)

    frame_idx = 0

    frames = []
    times = []
    times_ms = []

    mean_r = []
    mean_g = []
    mean_b = []
    red_strength = []

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y1:y2, x1:x2]

        b,g,r = cv2.split(roi_frame)

        r_mean = r.mean()
        g_mean = g.mean()
        b_mean = b.mean()

        strength = r_mean - (g_mean + b_mean)/2

        frames.append(frame_idx)
        times.append(frame_idx / fps)
        time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        times_ms.append(time_ms)

        mean_r.append(r_mean)
        mean_g.append(g_mean)
        mean_b.append(b_mean)
        red_strength.append(strength)

        frame_idx += 1

    cap.release()

    red_strength = np.array(red_strength)

    # ---------- adaptive threshold ----------
    lo = np.percentile(red_strength,20)
    hi = np.percentile(red_strength,80)

    thr_on = lo + 0.6*(hi-lo)
    thr_off = lo + 0.4*(hi-lo)

    led_state = []
    state = False
    state = red_strength[0] >= thr_on

    for val in red_strength:

        if not state and val >= thr_on:
            state = True
        elif state and val <= thr_off:
            state = False

        led_state.append(state)

    # ---------- dataframe ----------
    df = pd.DataFrame({
        "frame": frames,
        "time_sec": times,
        "time_ms": times_ms,
        "fps": fps,
        "mean_red": mean_r,
        "mean_green": mean_g,
        "mean_blue": mean_b,
        "red_strength": red_strength,
        "threshold_on": thr_on,
        "threshold_off": thr_off,
        "LED_on": led_state
    })

    csv_path = video_path.with_name(video_path.stem + "_LED_signal.csv")

    df.to_csv(csv_path, index=False)

    print("LED signal saved:", csv_path)

    return df

In [75]:
for video in videos:

    print("Processing:", video.name)

    roi = load_roi(video)

    df = extract_led_and_save(video, roi)

Processing: face_1440x1080_60_20260116_154145.mp4
Video: face_1440x1080_60_20260116_154145.mp4 FPS: 48.3702532368942
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\face_1440x1080_60_20260116_154145_LED_signal.csv
Processing: pupi_320x240_60_20260116_154145.mp4
Video: pupi_320x240_60_20260116_154145.mp4 FPS: 60.90751441794699
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\pupi_320x240_60_20260116_154145_LED_signal.csv
Processing: video_20260116_154140.mp4
Video: video_20260116_154140.mp4 FPS: 61.176735753949714
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\video_20260116_154140_LED_signal.csv


In [36]:
print(videos[2])

E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\video_20251216_165824.mp4


In [78]:
video = videos[1]
roi = load_roi(video)

df = extract_led_and_save(video, roi)

Video: pupi_320x240_60_20260116_154145.mp4 FPS: 60.90751441794699
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\pupi_320x240_60_20260116_154145_LED_signal.csv


In [76]:
import numpy as np
import pandas as pd
import cv2

def extract_led_intensity_and_save(video_path, roi):

    x1 = roi["x1"]
    y1 = roi["y1"]
    x2 = roi["x2"]
    y2 = roi["y2"]

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        raise RuntimeError("Could not read FPS from video")

    print("Video:", video_path.name, "FPS:", fps)

    frame_idx = 0

    frames = []
    times = []
    times_ms = []

    mean_r = []
    mean_g = []
    mean_b = []
    intensity_strength = []

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y1:y2, x1:x2]

        # split channels (for saving same columns)
        b, g, r = cv2.split(roi_frame)

        r_mean = r.mean()
        g_mean = g.mean()
        b_mean = b.mean()

        # compute grayscale intensity
        gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)
        intensity = gray.mean()

        frames.append(frame_idx)
        times.append(frame_idx / fps)

        time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        times_ms.append(time_ms)

        mean_r.append(r_mean)
        mean_g.append(g_mean)
        mean_b.append(b_mean)

        intensity_strength.append(intensity)

        frame_idx += 1

    cap.release()

    intensity_strength = np.array(intensity_strength)

    # ---------- adaptive threshold ----------
    lo = np.percentile(intensity_strength, 20)
    hi = np.percentile(intensity_strength, 80)

    thr_on = lo + 0.6 * (hi - lo)
    thr_off = lo + 0.4 * (hi - lo)

    led_state = []
    state = intensity_strength[0] >= thr_on

    for val in intensity_strength:

        if not state and val >= thr_on:
            state = True
        elif state and val <= thr_off:
            state = False

        led_state.append(state)

    # ---------- dataframe ----------
    df = pd.DataFrame({
        "frame": frames,
        "time_sec": times,
        "time_ms": times_ms,
        "fps": fps,
        "mean_red": mean_r,
        "mean_green": mean_g,
        "mean_blue": mean_b,
        "red_strength": intensity_strength,  # keep same column name
        "threshold_on": thr_on,
        "threshold_off": thr_off,
        "LED_on": led_state
    })

    csv_path = video_path.with_name(video_path.stem + "_LED_signal.csv")

    df.to_csv(csv_path, index=False)

    print("LED signal saved:", csv_path)

    return df

In [ ]:
video = videos[1]
roi = load_roi(video)

df = extract_led_intensity_and_save(video, roi)

Video: video_20251216_165824.mp4 FPS: 60.5282054234884
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\video_20251216_165824_LED_signal.csv
